In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'

data_path = project_path + '/data/raw/archive/data/data'
train_path = data_path + '/train'
test_path = data_path + '/test'

print("SERIAL 10 loaded successfully.")
print("Train path:", train_path)
print("Test path:", test_path)


SERIAL 10 loaded successfully.
Train path: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/train
Test path: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/data/raw/archive/data/data/test


In [5]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'

data_path = project_path + '/data/raw/archive/data/data'
train_path = data_path + '/train'
test_path = data_path + '/test'

print("Drive mounted successfully.")
print("Train folder exists:", os.path.exists(train_path))
print("Test folder exists:", os.path.exists(test_path))

Drive mounted successfully.
Train folder exists: True
Test folder exists: True


In [7]:
# Recreate channel summary

summary_data = []

for file in os.listdir(train_path):
    if file.endswith(".npy"):
        channel = file.replace(".npy", "")
        data = np.load(os.path.join(train_path, file))

        telemetry = data[:, 0]

        summary_data.append({
            "Channel": channel,
            "Time Steps": data.shape[0],
            "Features": data.shape[1],
            "Status": "Constant" if np.std(telemetry) == 0 else "Variable"
        })

channel_summary_df = pd.DataFrame(summary_data)

print("Channel summary recreated successfully.")
print("Total channels:", len(channel_summary_df))
print("\nFirst 10 channels:")
print(channel_summary_df.head(10))

Channel summary recreated successfully.
Total channels: 82

First 10 channels:
  Channel  Time Steps  Features    Status
0    E-13        2880        25  Variable
1     B-1        2435        25  Constant
2     T-9         439        55  Variable
3     D-7        2583        25  Constant
4    E-11        2880        25  Variable
5     D-2        2880        25  Constant
6    P-14        2880        55  Variable
7     E-3        2880        25  Variable
8     E-7        2769        25  Variable
9     P-7        2853        25  Variable


In [8]:
# Count constant and variable telemetry channels

status_counts = channel_summary_df["Status"].value_counts()

print("Telemetry channel status:")
print(status_counts)

print("\nPercentage:")
print((status_counts / len(channel_summary_df) * 100).round(2))

Telemetry channel status:
Status
Variable    68
Constant    14
Name: count, dtype: int64

Percentage:
Status
Variable    82.93
Constant    17.07
Name: count, dtype: float64


In [10]:
# Identify variable telemetry channels

variable_channels = channel_summary_df[
    channel_summary_df["Status"] == "Variable"
]["Channel"].tolist()

constant_channels = channel_summary_df[
    channel_summary_df["Status"] == "Constant"
]["Channel"].tolist()

print("Variable channels:", len(variable_channels))
print("Constant channels:", len(constant_channels))

print("\nFirst 10 variable channels:")
print(variable_channels[:10])

print("\nConstant channels:")
print(constant_channels)

Variable channels: 68
Constant channels: 14

First 10 variable channels:
['E-13', 'T-9', 'E-11', 'P-14', 'E-3', 'E-7', 'P-7', 'D-15', 'G-3', 'F-7']

Constant channels:
['B-1', 'D-7', 'D-2', 'D-8', 'D-13', 'C-2', 'D-9', 'D-12', 'P-4', 'M-6', 'G-2', 'S-2', 'D-14', 'T-5']


In [11]:
# Check telemetry ranges for variable channels

variable_range_summary = []

for channel in variable_channels:
    file_path = os.path.join(train_path, channel + ".npy")
    data = np.load(file_path)

    telemetry = data[:, 0]

    variable_range_summary.append({
        "Channel": channel,
        "Minimum": np.min(telemetry),
        "Maximum": np.max(telemetry),
        "Mean": np.mean(telemetry),
        "Std": np.std(telemetry)
    })

variable_range_df = pd.DataFrame(variable_range_summary)

print("Variable channel range summary:")
print(variable_range_df.head(10))

print("\nOverall telemetry range:")
print("Minimum:", variable_range_df["Minimum"].min())
print("Maximum:", variable_range_df["Maximum"].max())

Variable channel range summary:
  Channel  Minimum   Maximum      Mean       Std
0    E-13   -1.000  1.000000 -0.601640  0.545768
1     T-9   -1.000  1.000000  0.272594  0.202128
2    E-11   -1.000  1.000000 -0.587311  0.808742
3    P-14    0.999  1.000000  0.999490  0.000082
4     E-3   -1.000  1.000000  0.133693  0.416972
5     E-7   -1.000 -0.827000 -0.970138  0.043427
6     P-7   -1.000  1.000000  0.005532  0.240985
7    D-15   -1.000  1.191578  0.778429  0.655396
8     G-3   -1.000  1.000000  0.987805  0.155697
9     F-7   -1.000  1.000000 -0.608679  0.610538

Overall telemetry range:
Minimum: -1.47721668720188
Maximum: 4.162651279553374


In [12]:
# Find near-constant telemetry channels

near_constant_df = variable_range_df[
    variable_range_df["Std"] < 0.01
].sort_values("Std")

print("Channels with standard deviation < 0.01:")
print(near_constant_df.to_string(index=False))

print("\nNumber of near-constant variable channels:",
      len(near_constant_df))

Channels with standard deviation < 0.01:
Channel   Minimum   Maximum      Mean          Std
    A-1  0.999000  0.999000  0.999000 1.110223e-16
    R-1  0.999000  0.999000  0.999000 1.110223e-16
    A-5 -1.000000 -0.999000 -0.999710 7.441729e-05
   P-14  0.999000  1.000000  0.999490 8.192435e-05
    D-5  0.999000  1.000000  0.999396 2.272645e-04
   P-10  0.985882  1.001129  0.993367 1.241427e-03
    E-6  0.990000  1.000000  0.991278 3.338422e-03
    D-3 -1.000000 -0.970000 -0.984798 8.720074e-03
    G-4  0.980000  1.000000  0.994637 8.859715e-03
    D-4 -1.000000 -0.970000 -0.987370 9.390792e-03

Number of near-constant variable channels: 10


In [13]:
# Create the initial modeling channel list

usable_channels = variable_range_df[
    variable_range_df["Std"] >= 0.01
]["Channel"].tolist()

flagged_channels = variable_range_df[
    variable_range_df["Std"] < 0.01
]["Channel"].tolist()

print("Usable variable channels:", len(usable_channels))
print("Flagged near-constant channels:", len(flagged_channels))

print("\nFlagged channels:")
print(flagged_channels)

print("\nFirst 10 usable channels:")
print(usable_channels[:10])

Usable variable channels: 58
Flagged near-constant channels: 10

Flagged channels:
['P-14', 'D-4', 'D-3', 'A-1', 'P-10', 'A-5', 'E-6', 'D-5', 'R-1', 'G-4']

First 10 usable channels:
['E-13', 'T-9', 'E-11', 'E-3', 'E-7', 'P-7', 'D-15', 'G-3', 'F-7', 'E-10']


In [14]:
# Load telemetry signals for usable channels

train_telemetry = {}
test_telemetry = {}

for channel in usable_channels:
    train_file = os.path.join(train_path, channel + ".npy")
    test_file = os.path.join(test_path, channel + ".npy")

    train_data = np.load(train_file)
    test_data = np.load(test_file)

    train_telemetry[channel] = train_data[:, 0]
    test_telemetry[channel] = test_data[:, 0]

print("Training telemetry channels loaded:", len(train_telemetry))
print("Testing telemetry channels loaded:", len(test_telemetry))

print("\nExample:")
print("E-13 train shape:", train_telemetry["E-13"].shape)
print("E-13 test shape:", test_telemetry["E-13"].shape)

Training telemetry channels loaded: 58
Testing telemetry channels loaded: 58

Example:
E-13 train shape: (2880,)
E-13 test shape: (8640,)


In [15]:
# Compare training and testing telemetry ranges

scale_check = []

for channel in usable_channels:
    train_signal = train_telemetry[channel]
    test_signal = test_telemetry[channel]

    scale_check.append({
        "Channel": channel,
        "Train Min": np.min(train_signal),
        "Train Max": np.max(train_signal),
        "Test Min": np.min(test_signal),
        "Test Max": np.max(test_signal)
    })

scale_check_df = pd.DataFrame(scale_check)

print(scale_check_df.head(10))

print("\nOverall training range:")
print("Min:", scale_check_df["Train Min"].min())
print("Max:", scale_check_df["Train Max"].max())

print("\nOverall testing range:")
print("Min:", scale_check_df["Test Min"].min())
print("Max:", scale_check_df["Test Max"].max())

  Channel  Train Min  Train Max  Test Min  Test Max
0    E-13       -1.0   1.000000      -1.0  1.000000
1     T-9       -1.0   1.000000      -1.0  1.000000
2    E-11       -1.0   1.000000      -1.0  1.000000
3     E-3       -1.0   1.000000      -1.0  1.000000
4     E-7       -1.0  -0.827000      -1.0  1.000000
5     P-7       -1.0   1.000000      -1.0  1.000000
6    D-15       -1.0   1.191578      -1.0  1.375752
7     G-3       -1.0   1.000000      -1.0  1.000000
8     F-7       -1.0   1.000000      -1.0  1.000000
9    E-10       -1.0   1.000000      -1.0  1.000000

Overall training range:
Min: -1.47721668720188
Max: 4.162651279553374

Overall testing range:
Min: -1.4241973964125723
Max: 6.954363220802578


In [16]:
# Calculate preprocessing statistics using TRAINING data only

preprocessing_stats = []

for channel in usable_channels:
    train_signal = train_telemetry[channel]

    preprocessing_stats.append({
        "Channel": channel,
        "Train Mean": np.mean(train_signal),
        "Train Std": np.std(train_signal)
    })

preprocessing_stats_df = pd.DataFrame(preprocessing_stats)

print("Training preprocessing statistics calculated.")
print(preprocessing_stats_df.head(10))

Training preprocessing statistics calculated.
  Channel  Train Mean  Train Std
0    E-13   -0.601640   0.545768
1     T-9    0.272594   0.202128
2    E-11   -0.587311   0.808742
3     E-3    0.133693   0.416972
4     E-7   -0.970138   0.043427
5     P-7    0.005532   0.240985
6    D-15    0.778429   0.655396
7     G-3    0.987805   0.155697
8     F-7   -0.608679   0.610538
9    E-10   -0.587524   0.808322


In [17]:
# Standardize telemetry using TRAINING statistics only

preprocessed_train = {}
preprocessed_test = {}

for channel in usable_channels:
    train_signal = train_telemetry[channel]
    test_signal = test_telemetry[channel]

    stats = preprocessing_stats_df[
        preprocessing_stats_df["Channel"] == channel
    ].iloc[0]

    mean = stats["Train Mean"]
    std = stats["Train Std"]

    preprocessed_train[channel] = (train_signal - mean) / std
    preprocessed_test[channel] = (test_signal - mean) / std

print("Standardization completed.")

print("\nExample: E-13")
print("Original train mean:", np.mean(train_telemetry["E-13"]))
print("Processed train mean:", np.mean(preprocessed_train["E-13"]))

print("Original train std:", np.std(train_telemetry["E-13"]))
print("Processed train std:", np.std(preprocessed_train["E-13"]))

Standardization completed.

Example: E-13
Original train mean: -0.6016398712828454
Processed train mean: -1.0115365335473648e-16
Original train std: 0.5457678424622868
Processed train std: 0.9999999999999999


In [18]:
# Verify preprocessing on the test set

test_processed_stats = []

for channel in usable_channels:
    processed_test = preprocessed_test[channel]

    test_processed_stats.append({
        "Channel": channel,
        "Processed Test Mean": np.mean(processed_test),
        "Processed Test Std": np.std(processed_test)
    })

test_processed_stats_df = pd.DataFrame(test_processed_stats)

print("Processed test-set statistics:")
print(test_processed_stats_df.head(10))

Processed test-set statistics:
  Channel  Processed Test Mean  Processed Test Std
0    E-13            -0.003038            1.044206
1     T-9             1.856739            1.261634
2    E-11             0.033966            1.018558
3     E-3             0.866875            1.309128
4     E-7             0.128899            2.400404
5     P-7             0.161691            1.014608
6    D-15            -0.016366            1.014326
7     G-3             0.076377            0.147307
8     F-7             0.090550            1.078571
9    E-10             0.039078            1.027070


In [19]:
# Save training preprocessing statistics

preprocessing_stats_path = (
    project_path + '/results/preprocessing_statistics.csv'
)

preprocessing_stats_df.to_csv(
    preprocessing_stats_path,
    index=False
)

print("Preprocessing statistics saved successfully.")
print("Saved to:", preprocessing_stats_path)

Preprocessing statistics saved successfully.
Saved to: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/results/preprocessing_statistics.csv


In [20]:
# Final preprocessing verification

train_nan = sum(
    np.isnan(signal).sum()
    for signal in preprocessed_train.values()
)

test_nan = sum(
    np.isnan(signal).sum()
    for signal in preprocessed_test.values()
)

train_inf = sum(
    np.isinf(signal).sum()
    for signal in preprocessed_train.values()
)

test_inf = sum(
    np.isinf(signal).sum()
    for signal in preprocessed_test.values()
)

print("Processed training channels:", len(preprocessed_train))
print("Processed testing channels:", len(preprocessed_test))

print("\nTraining NaN values:", train_nan)
print("Testing NaN values:", test_nan)

print("Training infinite values:", train_inf)
print("Testing infinite values:", test_inf)

print(
    "\nStatistics file exists:",
    os.path.exists(preprocessing_stats_path)
)

Processed training channels: 58
Processed testing channels: 58

Training NaN values: 0
Testing NaN values: 0
Training infinite values: 0
Testing infinite values: 0

Statistics file exists: True


#github commite

In [21]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git status

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Refresh index: 100% (12/12), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/09_data_cleaning.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/10_data_preprocessing.ipynb
	results/preprocessing_statistics.csv

no changes added to commit (use "git add" and/or "git commit -a")


In [22]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git add notebooks/09_data_cleaning.ipynb
!git add notebooks/10_data_preprocessing.ipynb
!git add results/preprocessing_statistics.csv

!git commit -m "Complete SERIAL 10 data preprocessing"

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@ad4d591fa593.(none)')


In [23]:
!git config --global user.name "Amit Chandra Das"
!git config --global user.email "arickroy0@gmail.com"

print("Git identity configured.")

Git identity configured.


In [24]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git commit -m "Complete SERIAL 10 data preprocessing"

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
[main aa8c93a] Complete SERIAL 10 data preprocessing
 3 files changed, 61 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/10_data_preprocessing.ipynb
 create mode 100644 results/preprocessing_statistics.csv


In [25]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git push origin main

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 7.68 KiB | 786.00 KiB/s, done.
Total 7 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Amit-Chandra-Das/spacecraft-anomaly-detection.git
   36567ae..aa8c93a  main -> main
